# Machine Learning - Assignment 2

**Name:** Praveen Garg<br>
**BITS ID:** 2026AJ05478

## Predicting Online Shoppers' Purchasing Intention

This notebook uses the Online Shoppers Purchasing Intention dataset from the UCI Machine
Learning Repository to predict whether a browsing session on a shopping site ends in a
purchase (`Revenue = True`). It loads and checks the data, preprocesses it, and then
trains and evaluates five classification models. The trained models are saved in
`model/`, the notebook writes `test_data.csv` for the Streamlit app, and it ends with a
table comparing the five models.

In [1]:
# =============================================================================
# Libraries and Global Configuration
# =============================================================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)
from sklearn.model_selection import train_test_split

# Define a single fixed random seed used throughout this notebook.
# Fixing it makes the results reproducible, so the final run on the BITS
# Virtual Lab should give the same numbers as my local runs, as long as the
# library versions are the same on both.
RANDOM_STATE = 42

---
## 1. Problem statement

The goal is to predict, for one visit to an online shop, whether the session ends in a
purchase. Each session is described by 17 features. Ten are numeric: how many
administrative, informational and product-related pages the visitor opened and how long
they spent on each kind, three Google Analytics measures of those pages (bounce rate,
exit rate and page value), and how close the visit was to a special day. The other seven
are categorical and describe the visit: the month, operating system, browser, region,
traffic type, visitor type, and whether it fell on a weekend.

This is a binary classification problem. The positive class is a purchase
(`Revenue = True`) and the negative class is no purchase (`Revenue = False`). A shop
would want this prediction while a visitor is still on the site, so it can act on the
sessions that look unlikely to convert, for example with a discount, a live chat prompt
or product recommendations, without handing discounts to people who were going to buy
anyway.

---
## 2. The dataset

I use the Online Shoppers Purchasing Intention Dataset from the
[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset).
It is published under the
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence, which requires the
source to be credited:

- Creators: C. Sakar and Y. Kastro
- DOI: [10.24432/C5F88Q](https://doi.org/10.24432/C5F88Q)
- Paper: Sakar, C.O., Polat, S.O., Katircioglu, M. and Kastro, Y. (2019). *Real-time
  prediction of online shoppers' purchasing intention using multilayer perceptron and
  LSTM recurrent neural networks*. Neural Computing and Applications 31, 6893-6908.

Each row is one browsing session on an e-commerce site. According to the creators, the
data was put together so that each session belongs to a different user over a one-year
period, so that it does not lean towards any particular campaign, special day, user
profile or period.

In [2]:
# =============================================================================
# Load and Inspect Dataset
# =============================================================================

# Load the online shoppers intention dataset from the data directory
csv_file_path = 'data/online_shoppers_intention.csv'
shoppers_df = pd.read_csv(csv_file_path)

# 1. Dataset dimensions
print("=== Dataset Shape ===")
print(f"Shape: {shoppers_df.shape}\n")

# 2. Column data types
print("=== Column Data Types ===")
print(shoppers_df.dtypes)
print()

# 3. Missing values per column
print("=== Missing Values per Column ===")
missing_per_col = shoppers_df.isna().sum()
print(missing_per_col)
print(f"Total missing values: {missing_per_col.sum()}\n")

# 4. Duplicate rows and positive-revenue duplicates
duplicate_rows_mask = shoppers_df.duplicated()
duplicate_rows_count = duplicate_rows_mask.sum()
true_revenue_duplicates = (
    duplicate_rows_mask & shoppers_df['Revenue']
).sum()
print("=== Duplicate Rows Analysis ===")
print(f"Duplicate rows: {duplicate_rows_count}")
print(f"Duplicates with Revenue True: {true_revenue_duplicates}\n")

# 5. Target variable counts and proportion share
revenue_counts = shoppers_df['Revenue'].value_counts()
revenue_shares = shoppers_df['Revenue'].value_counts(normalize=True)
print("=== Target Distribution (Revenue) ===")
print(
    f"Revenue False: {revenue_counts[False]}, "
    f"Revenue True: {revenue_counts[True]}"
)
print(
    f"True share: {revenue_shares[True]:.3f} "
    f"({revenue_shares[True] * 100:.1f}%)"
)
print(
    f"False share: {revenue_shares[False]:.3f} "
    f"({revenue_shares[False] * 100:.1f}%)"
)

=== Dataset Shape ===
Shape: (12330, 18)

=== Column Data Types ===
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

=== Missing Values per Column ===
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues           

In [3]:
# =============================================================================
# Features and Target Separation
# =============================================================================

# Separate the predictor features (X) from the target outcome variable (y).
# The duplicates stay in for now; I deal with them in the preprocessing step.
X = shoppers_df.drop(columns=['Revenue'])
y = shoppers_df['Revenue']

# Print shapes of the feature matrix and target vector
print("=== Feature Matrix and Target Vector Shapes ===")
print(f"Feature matrix shape (X): {X.shape}")
print(f"Target vector shape (y): {y.shape}\n")

# Group the features by type: 10 numeric, 7 categorical (4 of them coded)
numeric_columns = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
categorical_columns = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]
coded_categorical_columns = [
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]

# Make sure the two lists together cover every feature exactly once,
# so no column gets missed or used twice later on
assert sorted(numeric_columns + categorical_columns) == sorted(X.columns)

print("=== Feature Type Breakdown ===")
print(f"Total features: {X.shape[1]}")
print(f"Numeric features: {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")
print(
    "Categorical features stored as number codes: "
    f"{len(coded_categorical_columns)}"
)

# Number of distinct values in each categorical column
print("\n=== Distinct Values per Categorical Column ===")
print(X[categorical_columns].nunique())

=== Feature Matrix and Target Vector Shapes ===
Feature matrix shape (X): (12330, 17)
Target vector shape (y): (12330,)

=== Feature Type Breakdown ===
Total features: 17
Numeric features: 10
Categorical features: 7
Categorical features stored as number codes: 4

=== Distinct Values per Categorical Column ===
Month               10
OperatingSystems     8
Browser             13
Region               9
TrafficType         20
VisitorType          3
Weekend              2
dtype: int64


**Dataset description**

The dataset has 12,330 rows, one per session, and 18 columns. Taking out the target,
`Revenue`, leaves 17 input features, which is well above the minimum of 12 features and
500 instances that the assignment asks for.

Ten of the features are numeric. Six describe the browsing itself: the number of
administrative, informational and product-related pages visited, and the time spent on
each of the three kinds. The other four are the Google Analytics page measures
(`BounceRates`, `ExitRates`, `PageValues`) and `SpecialDay`, which says how close the
visit was to a special day such as Valentine's Day.

The remaining seven are categorical. `Month` (10 distinct months) and `VisitorType`
(`Returning_Visitor`, `New_Visitor` or `Other`) are stored as text, and `Weekend` as
True/False. `OperatingSystems`, `Browser`, `Region` and `TrafficType` are stored as
integers, with 8, 13, 9 and 20 distinct codes, but the numbers are only labels for a
particular operating system, browser, region or traffic source. Browser 4 is not "more"
than browser 2, so these four have to be treated as categories and not as quantities.

The target `Revenue` is True/False, and the positive class is True, a purchase. The
classes are imbalanced: 10,422 sessions (84.5%) ended without a purchase and only 1,908
(15.5%) ended with one.

There are no missing values in any of the 18 columns. There are 125 duplicate rows,
meaning exact copies of an earlier row, and all 125 have `Revenue = False`.

The imbalance changes how I judge the models. A model that predicts "no purchase" for
every session would already get 84.5% accuracy while finding none of the buyers, so its
recall, F1 and MCC would all be 0. Accuracy on its own is misleading here. I still
report it, since the assignment asks for all six metrics, but I compare the models
mainly on recall, F1 and MCC, which only improve when buyers are actually found, and on
AUC, which shows how well a model ranks buyers above non-buyers.

---
## 3. Preprocessing

The 125 duplicate rows are all sessions that bounced immediately: one or two
product pages, a total time of 0 across all three duration columns, bounce and
exit rates of exactly 0.2 (the highest value either column takes), and a page
value of 0. The UCI description says each session belongs to a different
user, so these are most likely separate real visits that happen to produce
identical rows, not data-entry errors. I keep them.

I split the 10 numeric features into two groups based on skew. Seven columns
are heavily right-skewed (skew 1.96 to 7.58), so I apply log1p to compress the
long tail and then StandardScaler to centre and unit-scale them. I use log1p
rather than log because all seven have a minimum of 0, and log(0) is
undefined. The other three (BounceRates, ExitRates, SpecialDay) are capped at
small values (0.2 for the two rates, 1 for SpecialDay), so there are no
extreme values to pull in, and I only scale them.

The 7 categorical columns are one-hot encoded. Using `handle_unknown='ignore'`
means the encoder outputs all-zeros for any unseen category code at prediction
time, which protects the app from crashing on input it has not seen before.

Logistic regression, k-NN and SVM all use distance or gradient information and
are sensitive to feature scale, so they need the scaler. Decision trees split
on individual thresholds and are unaffected by scale. Gaussian Naive Bayes fits
a Gaussian on each feature independently and also does not need scaling,
although keeping it scaled does no harm.

In [4]:
# =============================================================================
# 3a. Inspect Duplicate Rows
# =============================================================================

# Pull out only the rows that are exact copies of an earlier row
dup_mask = shoppers_df.duplicated()
dup_df = shoppers_df[dup_mask]

# How many product-related pages did those sessions visit?
print("=== ProductRelated page counts in duplicates ===")
print(dup_df['ProductRelated'].value_counts())

# Did those sessions spend any time on pages at all?
dur_cols = [
    'Administrative_Duration',
    'Informational_Duration',
    'ProductRelated_Duration',
]
print("\n=== Duration column totals for duplicates ===")
print(dup_df[dur_cols].sum())

# Check the bounce/exit rates and page value for duplicates
print("\n=== BounceRates unique values in duplicates ===")
print(dup_df['BounceRates'].unique())
print("=== ExitRates unique values in duplicates ===")
print(dup_df['ExitRates'].unique())
print("=== PageValues unique values in duplicates ===")
print(dup_df['PageValues'].unique())

# Decision: keep the duplicate rows.
# All 125 are pure bounce sessions (one or two product pages, zero time on
# any page, bounce and exit rates of 0.2, zero page value). The UCI
# description says each session is a different user, so these are plausible
# real visits, and dropping them would under-count this kind of visit.

=== ProductRelated page counts in duplicates ===
ProductRelated
1    118
2      7
Name: count, dtype: int64

=== Duration column totals for duplicates ===
Administrative_Duration    0.0
Informational_Duration     0.0
ProductRelated_Duration    0.0
dtype: float64

=== BounceRates unique values in duplicates ===
[0.2]
=== ExitRates unique values in duplicates ===
[0.2]
=== PageValues unique values in duplicates ===
[0.]


In [5]:
# =============================================================================
# 3b. Skew Analysis of Numeric Columns
# =============================================================================

# Split the 10 numeric features into two groups:
#   skewed_cols  - heavily right-skewed, need log1p before scaling
#   bounded_cols - capped at small values, scaled directly
skewed_cols = [
    'Administrative',
    'Administrative_Duration',
    'Informational',
    'Informational_Duration',
    'ProductRelated',
    'ProductRelated_Duration',
    'PageValues',
]
bounded_cols = ['BounceRates', 'ExitRates', 'SpecialDay']

# Confirm the two groups together cover numeric_columns exactly
assert sorted(skewed_cols + bounded_cols) == sorted(numeric_columns), (
    "skewed_cols and bounded_cols must together equal numeric_columns"
)

# Print a compact table: original skew and skew after log1p transform
header = f"{'column':<30} {'min':>5} {'skew':>7} {'skew log1p':>12}"
print(header)
print("-" * len(header))
for col in skewed_cols:
    mn = shoppers_df[col].min()
    sk = shoppers_df[col].skew()
    sk_log = np.log1p(shoppers_df[col]).skew()
    print(f"{col:<30} {mn:>5.0f} {sk:>7.2f} {sk_log:>12.2f}")

# The bounded columns have no long tail to compress: show their range
print("\n=== Range of the bounded columns ===")
print(shoppers_df[bounded_cols].agg(['min', 'max']))

column                           min    skew   skew log1p
---------------------------------------------------------
Administrative                     0    1.96         0.57
Administrative_Duration            0    5.62         0.26
Informational                      0    4.04         2.03
Informational_Duration             0    7.58         1.92
ProductRelated                     0    4.34        -0.05
ProductRelated_Duration            0    7.26        -1.42
PageValues                         0    6.38         1.81

=== Range of the bounded columns ===
     BounceRates  ExitRates  SpecialDay
min          0.0        0.0         0.0
max          0.2        0.2         1.0


In [6]:
# =============================================================================
# 3c. Preprocessor Factory
# =============================================================================

# This is a function, not a single ColumnTransformer object, because each
# model in the next section needs its own fitted preprocessor inside its own
# Pipeline. If the same unfitted transformer object were reused, the second
# call to .fit() would overwrite the first, and each model's pipeline would
# share the same internal state.


def make_preprocessor():
    """Return a fresh, unfitted ColumnTransformer for this dataset."""
    # For the skewed columns: apply log1p first (using np.log1p directly,
    # not a lambda - lambdas cannot be pickled when saving the model),
    # then standardise.
    log_and_scale = Pipeline([
        ('log1p', FunctionTransformer(np.log1p)),
        ('scaler', StandardScaler()),
    ])

    return ColumnTransformer(
        transformers=[
            ('skewed', log_and_scale, skewed_cols),
            ('bounded', StandardScaler(), bounded_cols),
            # sparse_output=False keeps Gaussian NB from crashing;
            # handle_unknown='ignore' stops unseen codes crashing the app
            (
                'cat',
                OneHotEncoder(
                    handle_unknown='ignore',
                    sparse_output=False,
                ),
                categorical_columns,
            ),
        ]
    )

---
## 4. Train/test split and test_data.csv

I use one 80/20 split, stratified on `Revenue` so both parts keep the same
15.5% buyer rate as the full dataset, and seeded with `RANDOM_STATE` so it
comes out the same on every run. The same split is used for all five
models, so their test results are directly comparable. The test portion is
written to `test_data.csv` as raw, unprocessed data (the 17 feature columns
plus `Revenue`), because the saved models each carry their own fitted
preprocessor and apply it themselves when the app calls them.

In [7]:
# =============================================================================
# 4a. Train / Test Split
# =============================================================================

# One stratified split, seeded, used by every model in later cells
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("=== Split Shapes ===")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\n=== Buyer Share ===")
print(f"Train buyer share: {y_train.mean():.4f}")
print(f"Test  buyer share: {y_test.mean():.4f}")
print(f"Buyers in test set: {y_test.sum()}")

=== Split Shapes ===
X_train: (9864, 17)
X_test:  (2466, 17)
y_train: (9864,)
y_test:  (2466,)

=== Buyer Share ===
Train buyer share: 0.1547
Test  buyer share: 0.1549
Buyers in test set: 382


In [8]:
# =============================================================================
# 4b. Write test_data.csv and Verify Preprocessor Output Width
# =============================================================================

# Attach the target back to the raw test features before saving.
# The file stays in raw form so the app can feed it to any saved model,
# and the model's internal preprocessor handles encoding and scaling.
test_export = X_test.copy()
test_export['Revenue'] = y_test.values
test_export.to_csv('test_data.csv', index=False)

# Read it back and confirm it looks right
reloaded = pd.read_csv('test_data.csv')
print("=== test_data.csv shape ===")
print(reloaded.shape)

columns_match = list(reloaded.columns) == list(shoppers_df.columns)
print(f"\nColumn list matches original: {columns_match}")

dtypes_match = (reloaded.dtypes == shoppers_df.dtypes).all()
print(f"Dtypes match original:        {dtypes_match}")

# Fit one preprocessor on training data only (fitting on test data would
# leak test statistics into training), then check the output width
sample_pre = make_preprocessor()
sample_pre.fit(X_train)
train_transformed = sample_pre.transform(X_train)
test_transformed  = sample_pre.transform(X_test)

print("\n=== Preprocessor Output Shapes ===")
print(f"Train transformed: {train_transformed.shape}")
print(f"Test  transformed: {test_transformed.shape}")

=== test_data.csv shape ===


(2466, 18)

Column list matches original: True


Dtypes match original:        True



=== Preprocessor Output Shapes ===
Train transformed: (9864, 75)
Test  transformed: (2466, 75)
